In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score # Keeping AUC for individual model evaluation during CV
import gc # For garbage collection

# ================================
# Step 1: Load Data
# ================================
train = pd.read_parquet('train_data.parquet')
offers = pd.read_parquet('offer_metadata.parquet')
train['id3'] = train['id3'].astype(str).str.strip()
offers['id3'] = offers['id3'].astype(str).str.strip()
train = train.merge(offers, on='id3', how='left')

trans = pd.read_parquet("add_trans.parquet")

train['id5'] = pd.to_datetime(train['id5'], errors='coerce')
latest_date = train['id5'].max()
start_date_30d = latest_date - pd.Timedelta(days=30)
start_date_180d = latest_date - pd.Timedelta(days=180)

trans['f370'] = pd.to_datetime(trans['f370'], errors='coerce')
trans_180d = trans[(trans['f370'] >= start_date_180d) & (trans['f370'] <= latest_date)].copy()
trans_30d = trans_180d[trans_180d['f370'] >= start_date_30d].copy()

# ================================
# Step 2: Feature Engineering
# ================================
train['y'] = pd.to_numeric(train['y'], errors='coerce').fillna(0).astype(int)
train['id2'] = train['id2'].astype(str)
train['id3'] = train['id3'].astype(str)

feature_cols = []

# Offer-level smoothed CTR
global_ctr = train['y'].mean()
offer_agg = train.groupby('id3')['y'].agg(['sum', 'count'])
smoothing_factor = 25
offer_agg['offer_ctr_smoothed'] = (offer_agg['sum'] + global_ctr * smoothing_factor) / (offer_agg['count'] + smoothing_factor)
offer_agg.rename(columns={'count': 'offer_count'}, inplace=True)
# In your training script, after offer_agg is calculated and renamed:
offer_agg[['offer_ctr_smoothed', 'offer_count']].to_parquet('offer_agg.parquet', index=True)
# Also, ensure 'global_max_time' is saved:
global_max_time = train['id4'].max()
with open("global_max_time.pkl", "wb") as f:
    pickle.dump(global_max_time, f)
train = train.merge(offer_agg[['offer_ctr_smoothed', 'offer_count']], on='id3', how='left')
feature_cols.extend(['offer_ctr_smoothed', 'offer_count'])

# Time features
train['id4'] = pd.to_datetime(train['id4'], errors='coerce')
train['day_of_week'] = train['id5'].dt.dayofweek.fillna(-1).astype(int)
train['hour'] = train['id4'].dt.hour.fillna(-1).astype(int)
feature_cols.extend(['day_of_week', 'hour'])
train['user_offer_seen_count'] = train.groupby(['id2', 'id3'])['id3'].transform('size')
train['user_offer_rank'] = train.groupby('id2')['user_offer_seen_count'].rank(method='dense', ascending=False)
train['seen_bucket'] = pd.cut(train['user_offer_seen_count'], bins=[-1, 1, 3, 5, np.inf], labels=[0, 1, 2, 3]).astype(int)
feature_cols.extend(['user_offer_seen_count', 'user_offer_rank', 'seen_bucket'])
global_max_time = train['id4'].max()
train['last_seen'] = train.groupby(['id2', 'id3'])['id4'].transform('max')
train['days_since_seen'] = (global_max_time - train['last_seen']).dt.days.fillna(-1).astype(int)
train['recency_rank'] = train.groupby('id2')['last_seen'].rank(ascending=False)
user_agg = train.groupby('id2').agg(user_offer_count=('id3', 'nunique'), last_user_event=('id4', 'max'))
train = train.merge(user_agg, on='id2', how='left')
train['user_recency'] = (global_max_time - train['last_user_event']).dt.total_seconds() / (60*60*24)
train['user_offer_seen_freq'] = train['user_offer_seen_count'] / train['user_offer_count']
feature_cols.extend(['days_since_seen', 'recency_rank', 'user_offer_count', 'user_recency', 'user_offer_seen_freq'])
train.drop(columns=['last_seen', 'last_user_event'], inplace=True)
f_cols = [f"f{i}" for i in range(1, 367)]
categorical_f = ['f42', 'f48', 'f50', 'f52', 'f53', 'f54', 'f55', 'f56', 'f57', 'f349', 'f354']
numerical_f = list(set(f_cols) - set(categorical_f))
label_encoders = {}
for col in categorical_f:
    if col in train.columns:
        train[col] = train[col].fillna("missing").astype(str)
        le = LabelEncoder()
        all_known_labels = train[col].unique().tolist()
        if "missing" not in all_known_labels:
             all_known_labels.append("missing")
        le.fit(all_known_labels)
        train[col] = le.transform(train[col])
        label_encoders[col] = le
with open("f_series_label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)
for col in numerical_f:
    if col in train.columns:
        train[col] = pd.to_numeric(train[col], errors='coerce').fillna(-1).astype('float32')
processed_f_cols = categorical_f + numerical_f
feature_cols = [col for col in feature_cols if col not in f_cols]
feature_cols.extend(processed_f_cols)
category_30 = [f'f{i}' for i in range(152, 163)]
category_180 = [f'f{i}' for i in range(163, 174)]
train['total_spend_30'] = train[category_30].sum(axis=1)
train['total_spend_180'] = train[category_180].sum(axis=1)
train[[col + '_ratio_30' for col in category_30]] = train[category_30].div(train['total_spend_30'] + 1e-6, axis=0)
train[[col + '_ratio_180' for col in category_180]] = train[category_180].div(train['total_spend_180'] + 1e-6, axis=0)
train['top_spend_30'] = train[category_30].idxmax(axis=1)
train['top_spend_180'] = train[category_180].idxmax(axis=1)
train['top_spend_30'] = train['top_spend_30'].fillna("missing_category").astype(str)
train['top_spend_180'] = train['top_spend_180'].fillna("missing_category").astype(str)
short_term_spend = train[category_30]
long_term_spend = train[category_180]
long_term_spend.columns = short_term_spend.columns
train[[f'shift_{col}' for col in category_30]] = short_term_spend / (long_term_spend + 1e-6)
top_spend_encoder = LabelEncoder()
all_categories = np.union1d(train['top_spend_30'].unique(), train['top_spend_180'].unique())
if "missing_category" not in all_categories:
    all_categories = np.append(all_categories, "missing_category")
top_spend_encoder.fit(all_categories)
train['top_spend_30_encoded'] = top_spend_encoder.transform(train['top_spend_30'])
train['top_spend_180_encoded'] = top_spend_encoder.transform(train['top_spend_180'])
with open("top_spend_encoder.pkl", "wb") as f:
    pickle.dump(top_spend_encoder, f)
feature_cols.extend(
    ['top_spend_30_encoded', 'top_spend_180_encoded', 'total_spend_30', 'total_spend_180']
    + [f + '_ratio_30' for f in category_30]
    + [f + '_ratio_180' for f in category_180]
    + [f'shift_{f}' for f in category_30]
)
feature_cols = sorted(list(set(feature_cols)))
feature_cols = [col for col in feature_cols if col in train.columns]

# ================================
# Step 5: Train Model (Improved)
# ================================
train = train.dropna(subset=['id5']).sort_values('id5', ascending=True)
X = train[feature_cols]
y = train['y']
lgb_categorical_features = [col for col in categorical_f if col in X.columns]
lgb_categorical_features.extend(['day_of_week', 'hour', 'seen_bucket', 'top_spend_30_encoded', 'top_spend_180_encoded'])
lgb_categorical_features = list(set([col for col in lgb_categorical_features if col in X.columns]))
split_point = int(len(train) * 0.8)
X_train_split, X_val_split = X.iloc[:split_point], X.iloc[split_point:]
y_train_split, y_val_split = y.iloc[:split_point], y.iloc[split_point:]
train_id2_val = train.loc[X_val_split.index, 'id2'].values
train_id3_val = train.loc[X_val_split.index, 'id3'].values
pos_weight = (y_train_split == 0).sum() / (y_train_split == 1).sum()
params = {
    'objective': 'binary',
    'metric': 'auc', # Keep AUC for early stopping and general evaluation
    'boosting_type': 'gbdt',
    'n_estimators': 2000, # Increase n_estimators and rely on early stopping
    'learning_rate': 0.01, # Reduce learning rate
    'num_leaves': 96, # Increase num_leaves for more complexity
    'max_depth': 8, # Set a max_depth to prevent overfitting (or keep -1 if you're sure)
    'lambda_l1': 0.1, # L1 regularization to encourage sparsity
    'lambda_l2': 0.1, # L2 regularization to prevent large weights
    'min_child_samples': 20, # Minimum number of data needed in a child (leaf)
    'subsample': 0.7, # Subsample ratio of the training instance
    'colsample_bytree': 0.7, # Subsample ratio of columns when constructing each tree
    'scale_pos_weight': pos_weight,
    'verbose': -1,
    'n_jobs': -1, # Use all available cores
    'seed': 42, # For reproducibility
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'data_random_seed': 42,
}
print(f"Number of features: {len(feature_cols)}")
print(f"Categorical features for LightGBM: {lgb_categorical_features}")
model = lgb.LGBMClassifier(**params) # Use LGBMClassifier for scikit-learn API compatibility
model.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    eval_metric='auc', # Can also monitor 'binary_logloss' or others
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)], # Increased stopping rounds
    categorical_feature=lgb_categorical_features
)
y_pred_val = model.predict_proba(X_val_split)[:, 1]
eval_df = pd.DataFrame({
    'id2': train_id2_val,
    'id3': train_id3_val,
    'y': y_val_split.values,
    'y_pred': y_pred_val
})
def apk(actual, predicted, k=7):
    if len(predicted) > k:
        predicted = predicted[:k]
    score, num_hits = 0.0, 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k) if actual else 0.0
def mapk(actual_list, predicted_list, k=7):
    return np.mean([apk(a, p, k) for a, p in zip(actual_list, predicted_list)])
actual_list_val, predicted_list_val = [], []
for _, group in eval_df.groupby('id2'):
    actual = set(group.loc[group['y'] == 1, 'id3'].tolist())
    top_k = group.sort_values('y_pred', ascending=False).head(7)['id3'].tolist()
    actual_list_val.append(actual)
    predicted_list_val.append(top_k)
score_val = mapk(actual_list_val, predicted_list_val, k=7)
print(f"✅ MAP@7 score (Time-Based Validation - Tuned): {score_val:.5f}")
print("\n--- Starting Time-Series K-Fold Cross-Validation ---")
n_splits = 5 # Number of folds for CV
kf = KFold(n_splits=n_splits, shuffle=False) # Important: shuffle=False for time-series
oof_preds = np.zeros(len(train)) # Out-of-fold predictions for meta-modeling or robust evaluation
models = [] # To store trained models from each fold
fold_map_scores = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/{n_splits} ---")
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    train_id2_val_fold = train.loc[X_val_fold.index, 'id2'].values
    train_id3_val_fold = train.loc[X_val_fold.index, 'id3'].values
    pos_weight_fold = (y_train_fold == 0).sum() / (y_train_fold == 1).sum()
    current_params = params.copy()
    current_params['scale_pos_weight'] = pos_weight_fold
    model_fold = lgb.LGBMClassifier(**current_params)
    model_fold.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)], # Silent verbose during CV
        categorical_feature=lgb_categorical_features
)
    models.append(model_fold) # Store model
    fold_preds = model_fold.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = fold_preds
    eval_df_fold = pd.DataFrame({
        'id2': train_id2_val_fold,
        'id3': train_id3_val_fold,
        'y': y_val_fold.values,
        'y_pred': fold_preds
    })
    actual_list_fold, predicted_list_fold = [], []
    for _, group in eval_df_fold.groupby('id2'):
        actual = set(group.loc[group['y'] == 1, 'id3'].tolist())
        top_k = group.sort_values('y_pred', ascending=False).head(7)['id3'].tolist()
        actual_list_fold.append(actual)
        predicted_list_fold.append(top_k)
    fold_map7_score = mapk(actual_list_fold, predicted_list_fold, k=7)
    fold_map_scores.append(fold_map7_score)
    print(f"Fold {fold + 1} MAP@7: {fold_map7_score:.5f}")
print(f"\nAverage CV MAP@7 score: {np.mean(fold_map_scores):.5f} +/- {np.std(fold_map_scores):.5f}")
print("\n--- Training Final Model on Full Data ---")
final_model = lgb.LGBMClassifier(**params) # Use the same tuned parameters
final_model.fit(
    X, y,
    categorical_feature=lgb_categorical_features
)
with open("lgbm_model.pkl", "wb") as f:
    pickle.dump(final_model, f)
print("✅ Final LightGBM model saved as lgbm_model.pkl")
with open("feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)
print("✅ Feature columns saved as feature_cols.pkl")
del train, X, y, X_train_split, X_val_split, y_train_split, y_val_split, eval_df, actual_list_val, predicted_list_val
del X_train_fold, X_val_fold, y_train_fold, y_val_fold, eval_df_fold, actual_list_fold, predicted_list_fold
gc.collect()

C:\Users\dhruv\AppData\Local\Temp\ipykernel_16724\1159806266.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['total_spend_30'] = train[category_30].sum(axis=1)
C:\Users\dhruv\AppData\Local\Temp\ipykernel_16724\1159806266.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['total_spend_180'] = train[category_180].sum(axis=1)
C:\Users\dhruv\AppData\Local\Temp\ipykernel_16724\1159806266.py:99: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

Number of features: 415
Categorical features for LightGBM: ['f57', 'f53', 'f56', 'f50', 'f48', 'top_spend_30_encoded', 'day_of_week', 'f54', 'f349', 'f52', 'seen_bucket', 'top_spend_180_encoded', 'f42', 'hour', 'f354', 'f55']
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[638]	valid_0's auc: 0.924913
Early stopping, best iteration is:
[638]	valid_0's auc: 0.924913
✅ MAP@7 score (Time-Based Validation - Tuned): 0.05031

--- Starting Time-Series K-Fold Cross-Validation ---

--- Fold 1/5 ---
✅ MAP@7 score (Time-Based Validation - Tuned): 0.05031

--- Starting Time-Series K-Fold Cross-Validation ---

--- Fold 1/5 ---


KeyboardInterrupt: 